# mini_sim_db tutorial

This notebook explains the full idea behind `mini_sim_db` and shows practical usage beyond the short README.

## 1) Design idea

`mini_sim_db` is intentionally simple:

- Store simulation run metadata in a local SQLite file.
- Keep a tiny local CLI for direct use in submit/post scripts.
- Make **local-first** recording the default.
- Support **later sync** via portable JSON sync artifacts.
- Keep remote host/client transport available as an **optional** module under `remote_api/`.

This keeps the core workflow robust even when machines cannot directly reach each other.


## 2) Schema and fields

The main storage is SQLite, with one primary row per simulation case/job and stable fields such as:

- `case`: case ID
- `job_id`: deterministic hash for the logical job identity
- `work_dir`: run working directory
- `bin`: solver/executable label
- `inp`: primary input file
- `input_files`: all input files joined with `;`
- `extra_params`: extra runtime parameters (usually JSON text)
- `run_host`: host that initiated the write
- `status`: `start`, `restart`, or `done`
- `note`: short human note
- `notes`: legacy compatibility field
- `state_changed_at`: timestamp for status lifecycle events
- `created_at`: initial creation timestamp
- `updated_at`: last update timestamp

### Compatibility

- `.csv` paths are still accepted for compatibility.
- A `.csv` path maps to a sibling `.sqlite3` database.
- If a legacy CSV exists and the SQLite DB does not yet exist, rows are auto-imported on first open.


## 3) Local-first workflow (CLI)

This is now the primary workflow.

Use the local CLI to record, inspect, and update jobs on each machine independently.


In [ ]:
# init
python sim_db.py init

# add a case with extra params
python sim_db.py add \
  --case wing_040 \
  --work-dir /scratch/wing_040 \
  --inp wing_040.inp \
  --bin solver_v2 \
  --extra-param threads=16 \
  --extra-param precision=double \
  --status start

# done + list
python sim_db.py done --case wing_040
python sim_db.py list

## 4) Sync workflow (portable JSON artifact)

Use this when machines cannot keep a live connection to one central host.

Export local updates from one machine and import them on another.


In [ ]:
# terminal A: server
export SIM_DB_API_TOKEN='replace-me'
python sim_db_server.py --host 127.0.0.1 --port 8765 --db ~/sim_db.csv

In [ ]:
# terminal B: client
export SIM_DB_API_TOKEN='replace-me'
python sim_db_client.py --url http://127.0.0.1:8765 init
python sim_db_client.py --url http://127.0.0.1:8765 create \
  --case c100 --inp c100.inp --bin solver --status start \
  --extra-params '{"threads":8,"mode":"fast"}'
python sim_db_client.py --url http://127.0.0.1:8765 done --case c100
python sim_db_client.py --url http://127.0.0.1:8765 list

### Sync behavior and `run_host`

Sync is based on portable JSON artifacts and uses:

- `job_id` as the merge key
- `updated_at` as the primary freshness signal

Current reconcile policy:

1. incoming newer than local -> overwrite local row
2. equal timestamps -> skip as unchanged
3. local newer than incoming -> keep local and report conflict
4. same case name used by a different job -> report conflict

`run_host` still records which machine originated a write, which helps later inspection and reconciliation.


## 5) Optional remote transport and security notes

Remote client/server transport is still available, but it is now optional and lives under `remote_api/`.

If you use it:

- Always set and require `SIM_DB_API_TOKEN` (or pass `--token`).
- Bind to loopback (`127.0.0.1`) unless you explicitly need remote access.
- If exposing remotely, use firewall/Tailscale/VPN and strict path allowlists.
- Prefer environment variables for tokens over hardcoding them in scripts.
- Keep `--allowed-db-path` / `--allowed-base-dir` tight on server side.

If direct connectivity is not practical, prefer the local-first sync workflow instead of forcing live exposure.


## 6) Practical examples

### Example A: submit script integration

Record `start` before submission, then `done` after post-processing.

In [ ]:
#!/usr/bin/env bash
set -euo pipefail

CASE=case_123
DB=${DB:-$HOME/sim_db.csv}

python sim_db.py init --db "$DB"
python sim_db.py add --case "$CASE" --inp "${CASE}.inp" --bin solver --status start --db "$DB"
# ... submit workload ...
python sim_db.py done --case "$CASE" --db "$DB"

### Example B: track restarts clearly

Use `status=restart` with a short note so later analysis is easier.

In [ ]:
python sim_db.py add \
  --case case_777 \
  --inp case_777.inp \
  --bin solver_v3 \
  --status restart \
  --note 'rerun after mesh repair'

---

If you only need the quick commands, go back to `README.md`.
If you need structure and context, keep this notebook as your reference.